In [48]:
#data manipulation
import pandas as pd
import numpy as np

#viz
import seaborn as sns
import matplotlib.pyplot as plt

#NLP
import re
import nltk
import string
from nltk import sent_tokenize, word_tokenize
from nltk.tokenize.toktok import ToktokTokenizer
from bs4 import BeautifulSoup
import spacy
from nltk.stem import WordNetLemmatizer

%matplotlib inline

#MISC
import warnings
warnings.filterwarnings('ignore')
import unicodedata
from tqdm import tqdm

In [49]:
#Reading  data files
train = pd.read_csv(r"train.csv")
test = pd.read_csv(r"test.csv")

In [50]:
test.head(20)

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.
5,0001ea8717f6de06,Thank you for understanding. I think very high...
6,00024115d4cbde0f,Please do not add nonsense to Wikipedia. Such ...
7,000247e83dcc1211,:Dear god this site is horrible.
8,00025358d4737918,""" \n Only a fool can believe in such numbers. ..."
9,00026d1092fe71cc,== Double Redirects == \n\n When fixing double...


In [51]:
class_names = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [52]:
train_text = train['comment_text']
test_text = test['comment_text']
all_text = pd.concat([train_text, test_text])

In [53]:
%%time
from sklearn.feature_extraction.text import TfidfVectorizer
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english',
    ngram_range=(1, 1),
    max_features=10000)
word_vectorizer.fit(all_text)
train_word_features = word_vectorizer.transform(train_text)
test_word_features = word_vectorizer.transform(test_text)

Wall time: 56.6 s


In [54]:
%%time
char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char',
    stop_words='english',
    ngram_range=(2, 6),
    max_features=50000)
char_vectorizer.fit(all_text)
train_char_features = char_vectorizer.transform(train_text)
test_char_features = char_vectorizer.transform(test_text)

KeyboardInterrupt: 

In [55]:
from scipy.sparse import hstack
train_features = hstack([train_char_features, train_word_features])
test_features = hstack([test_char_features, test_word_features])

NameError: name 'train_char_features' is not defined

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
scores = []
submission = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = train[class_name]
    classifier = LogisticRegression(C=0.1, solver='sag')

    cv_score = np.mean(cross_val_score(classifier, train_features, train_target, cv=3, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(train_features, train_target)
    submission[class_name] = classifier.predict_proba(test_features)[:, 1]

print('Total CV score is {}'.format(np.mean(scores)))

submission.to_csv('submission.csv', index=False)

# My Version

# Normalizing Comments

In [56]:
import string
import re
from nltk.tokenize import word_tokenize
TAG_RE = re.compile(r'<[^>]+>') # remove <br> and other html tags

def clean_text(text):
    # lower text
    text = text.lower()

    # Reemove Html Tags like <br>
    text= TAG_RE.sub('', text)

    # remove puncutation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenize_text
    text = [word for word in word_tokenize(text)]

    # remove words that contain numbers
    text = [word for word in text if not any(c.isdigit() for c in word)]
    
    # remove empty tokens
    text = [t for t in text if len(t) > 0]

    # remove words with only one letter
    text = [t for t in text if len(t) > 1]

    # join all
    text = " ".join(text)
    return(text)

In [57]:
train["clean_comments"] = train["comment_text"].apply(lambda x:clean_text(x))

test["clean_comments"] = test["comment_text"].apply(lambda x:clean_text(x))

In [58]:
# removing html tags
def strip_html_tags(text):
    soup = BeautifulSoup(text,'html.parser')
    stripped_text = soup.get_text()
    return stripped_text

In [59]:
# removing accented characters
def remove_accented_chars(text):
    text = unicodedata.normalize('NFKD',text).encode('ascii','ignore').decode('utf-8','ignore')
    return text

!pip install contractions
from contractions import contractions_dict
# Expand contractions
def expand_contractions(text, contraction_mapping=contractions_dict):
    contractions_pattern = re.compile('({})'.format('|'.join(contraction_mapping.keys())),
                                      flags=re.IGNORECASE|re.DOTALL)
    
    def expand_match(contraction):
        match = contraction.group(0)
        first_char = match[0]
        expanded_contraction = contraction_mapping.get(match)\
                                if contraction_mapping.get(match)\
                                else contraction_mapping.get(match.lower())
        expanded_contraction = first_char + expanded_contraction[1:]
        return expanded_contraction
    
    expanded_text = contractions_pattern.sub(expand_match, text)
    expanded_text = re.sub("'","",expanded_text)
    return expanded_text

In [60]:
nlp = spacy.load('en_core_web_sm', parse=True, tag=True, entity=True)
tokenizer = ToktokTokenizer()
stopword_list = nltk.corpus.stopwords.words('english')

In [61]:
# Lemmatization
def lemmatize_text(text):
    text = nlp(text)
    text = ' '.join([word.lemma_ if word.lemma_ !='-PRON-' else word.text for word in text])
    return text

In [62]:
# Removing special characters
def remove_special_characters(text, remove_digits):
    pattern = r'[^a-zA-Z0-9\s]' if not remove_digits else r'[^a-zA-Z\s]'
    text = re.sub(pattern,'',text)
    return text

In [63]:
# Removing stopwords

def remove_stopwords(text, is_lower_case):
    tokens = tokenizer.tokenize(text)
    tokens = [token.strip() for token in tokens]
    if is_lower_case:
        filtered_tokens = [token for token in tokens if token not in stopword_list]
    else:
        filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text

In [64]:
# Normalized corpus
def normalize_corpus(corpus,html_stripping=True,
                      accented_char_removal=True, text_lower_case=True,
                      text_lemmatization=True, special_char_removal=True,
                      stopword_removal=True):
    
    normalized_corpus = []
    try:
        for doc in corpus:
            # strip HTML
            if html_stripping:
                doc = strip_html_tags(doc)
            # remove accented characters
            if accented_char_removal:
                doc = remove_accented_chars(doc)
            # lowercase the text
            if text_lower_case:
                doc = doc.lower()
            # remove extra newlines
            doc = re.sub(r'[\r|\n\r\n]+','',doc)
            # lemmatize text
            if text_lemmatization:
                doc = lemmatize_text(doc)
            # remove special characters or digits
            if special_char_removal:
                # insert spaces between special character to isolate them
                special_char_pattern = re.compile(r'([{.(-)!}])')
                doc = special_char_pattern.sub(" \\1 ",doc)
                doc = remove_special_characters(doc, True)
            # remove extra whitespace
            doc = re.sub(' +', ' ', doc)
            # remove stopwords
            if stopword_removal:
                doc = remove_stopwords(doc, is_lower_case=text_lower_case)
        
            normalized_corpus.append(doc)
     
        return normalized_corpus 
    except:
        return normalized_corpus

In [65]:
train.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,clean_comments
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,explanation why the edits made under my userna...
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,daww he matches this background colour im seem...
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,hey man im really not trying to edit war its j...
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,more cant make any real suggestions on improve...
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,you sir are my hero any chance you remember wh...


In [66]:
%%time
train_norm_corpus = normalize_corpus(train["clean_comments"],html_stripping=True,
                      accented_char_removal=True, text_lower_case=True,
                      text_lemmatization=False, special_char_removal=True,
                      stopword_removal=True)

test_norm_corpus = normalize_corpus(test["clean_comments"],html_stripping=True,
                      accented_char_removal=True, text_lower_case=True,
                      text_lemmatization=False, special_char_removal=True,
                      stopword_removal=True)

Wall time: 2min 27s


In [67]:
train_norm_corpus[0:5]

['explanation edits made username hardcore metallica fan reverted werent vandalisms closure gas voted new york dolls fac please dont remove template talk page since im retired',
 'daww matches background colour im seemingly stuck thanks talk january utc',
 'hey man im really trying edit war guy constantly removing relevant information talking edits instead talk page seems care formatting actual info',
 'cant make real suggestions improvement wondered section statistics later subsection types accidents think references may need tidying exact format ie date format etc later noone else first preferences formatting style references want please let know appears backlog articles review guess may delay reviewer turns listed relevant form eg wikipediagoodarticlenominationstransport',
 'sir hero chance remember page thats']

In [68]:
# Adding normalized data to dataframe
train['Normalized_Comments'] = pd.Series(train_norm_corpus)
test['Normalized_Comments'] = pd.Series(test_norm_corpus)

In [69]:
min(train['Normalized_Comments'].apply(len)),min(test['Normalized_Comments'].apply(len))

(0, 0)

In [70]:
train = train.replace(r'^(\s?)+$',"blank",regex=True)
test = test.replace(r'^(\s?)+$',"blank",regex=True)
train.info(),test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   id                   159571 non-null  object
 1   comment_text         159571 non-null  object
 2   toxic                159571 non-null  int64 
 3   severe_toxic         159571 non-null  int64 
 4   obscene              159571 non-null  int64 
 5   threat               159571 non-null  int64 
 6   insult               159571 non-null  int64 
 7   identity_hate        159571 non-null  int64 
 8   clean_comments       159571 non-null  object
 9   Normalized_Comments  159571 non-null  object
dtypes: int64(6), object(4)
memory usage: 12.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   id                   15

(None, None)

In [71]:
train = train[train['Normalized_Comments'].notna()]
train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159571 entries, 0 to 159570
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   id                   159571 non-null  object
 1   comment_text         159571 non-null  object
 2   toxic                159571 non-null  int64 
 3   severe_toxic         159571 non-null  int64 
 4   obscene              159571 non-null  int64 
 5   threat               159571 non-null  int64 
 6   insult               159571 non-null  int64 
 7   identity_hate        159571 non-null  int64 
 8   clean_comments       159571 non-null  object
 9   Normalized_Comments  159571 non-null  object
dtypes: int64(6), object(4)
memory usage: 13.4+ MB


In [72]:
train.reset_index(inplace= True)
train[-5:]

,index,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,clean_comments,Normalized_Comments
159566,159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0,and for the second time of asking when your vi...,second time asking view completely contradicts...
159567,159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0,you should be ashamed of yourself that is horr...,ashamed horrible thing put talk page
159568,159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0,spitzer umm theres no actual article for prost...,spitzer umm theres actual article prostitution...
159569,159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0,and it looks like it was actually you who put ...,looks like actually put speedy first version d...
159570,159570,fff46fc426af1f9a,"""\nAnd ... I really don't think you understand...",0,0,0,0,0,0,and really dont think you understand came here...,really dont think understand came idea bad rig...


# TF-IDF features with Classification Models

In [95]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [96]:
#Build TF-IDF features on train articles
tv = TfidfVectorizer(use_idf=True, min_df=0.0, max_df=1.0)
tv_train_features = tv.fit_transform(train['Normalized_Comments'])

In [97]:
# transform test articles into features
tv_test_features = tv.transform(test['Normalized_Comments'])

In [98]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
scores = []
submission = pd.DataFrame.from_dict({'id': test['id']})
submission_2 = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = train[class_name]
    classifier = LogisticRegression(C=0.1, solver='sag')

    cv_score = np.mean(cross_val_score(classifier, tv_train_features, train_target, cv=5, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(tv_train_features, train_target)
    submission[class_name] = classifier.predict_proba(tv_test_features)[:, 1]
    submission_2[class_name] = classifier.predict(tv_test_features)

print('Total CV score is {}'.format(np.mean(scores)))

submission.to_csv('submission.csv', index=False)

CV score for class toxic is 0.955666027022399
CV score for class severe_toxic is 0.984029302681874
CV score for class obscene is 0.9776917045078439
CV score for class threat is 0.9756436465520422
CV score for class insult is 0.9684874425261023
CV score for class identity_hate is 0.9668538514454097
Total CV score is 0.971395329122612


In [99]:
from sklearn.svm import LinearSVC
scores = []
submission_svc = pd.DataFrame.from_dict({'id': test['id']})
submission_2_svc = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = train[class_name]
    classifier =LinearSVC(penalty='l2', C=1, random_state=42)

    cv_score = np.mean(cross_val_score(classifier, tv_train_features, train_target, cv=5, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(tv_train_features, train_target)
    submission_svc[class_name] = classifier.decision_function(tv_test_features)
    submission_2_svc[class_name] = classifier.predict(tv_test_features)

print('Total CV score is {}'.format(np.mean(scores)))

submission_svc.to_csv('submission_svc.csv', index=False)

CV score for class toxic is 0.964605846054243
CV score for class severe_toxic is 0.966474157208461
CV score for class obscene is 0.9793289812164849
CV score for class threat is 0.9685921385589886
CV score for class insult is 0.9663300321554535
CV score for class identity_hate is 0.9572202840342371
Total CV score is 0.967091906537978


# With N- Grams

In [100]:
#Build TF-IDF features on train articles
tv_1_2 = TfidfVectorizer(use_idf=True, min_df=0.0, max_df=1.0,ngram_range=(1,2))
tv_train_features = tv_1_2.fit_transform(train['Normalized_Comments'])

In [101]:
#Build TF-IDF features on train articles
# transform test articles into features
tv_test_features = tv_1_2.transform(test['Normalized_Comments'])

In [102]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
scores = []
submission = pd.DataFrame.from_dict({'id': test['id']})
submission_2 = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = train[class_name]
    classifier = LogisticRegression(C=0.1, solver='sag')

    cv_score = np.mean(cross_val_score(classifier, tv_train_features, train_target, cv=5, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(tv_train_features, train_target)
    submission[class_name] = classifier.predict_proba(tv_test_features)[:, 1]
    submission_2[class_name] = classifier.predict(tv_test_features)

print('Total CV score is {}'.format(np.mean(scores)))

submission.to_csv('submission_2_lr.csv', index=False)

CV score for class toxic is 0.9489666871660933
CV score for class severe_toxic is 0.983770287497069
CV score for class obscene is 0.9744543702837998
CV score for class threat is 0.976385886515371
CV score for class insult is 0.9652568249193608
CV score for class identity_hate is 0.9645617817960023
Total CV score is 0.9688993063629493


In [103]:
from sklearn.svm import LinearSVC
scores = []
submission_svc = pd.DataFrame.from_dict({'id': test['id']})
submission_2_svc = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = train[class_name]
    classifier =LinearSVC(penalty='l2', C=1, random_state=42)

    cv_score = np.mean(cross_val_score(classifier, tv_train_features, train_target, cv=5, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(tv_train_features, train_target)
    submission_svc[class_name] = classifier.decision_function(tv_test_features)
    submission_2_svc[class_name] = classifier.predict(tv_test_features)

print('Total CV score is {}'.format(np.mean(scores)))

submission_svc.to_csv('submission_2_svc.csv', index=False)

CV score for class toxic is 0.970538709493012
CV score for class severe_toxic is 0.9741362583956723
CV score for class obscene is 0.984479666415804
CV score for class threat is 0.9765291053896525
CV score for class insult is 0.9745698803187588
CV score for class identity_hate is 0.9656186268265113
Total CV score is 0.974312041139902


# Using Clean Comments

In [104]:
#Build TF-IDF features on train articles
tv_1_2 = TfidfVectorizer(use_idf=True, min_df=0.0, max_df=1.0,ngram_range=(1,2))
tv_train_features = tv_1_2.fit_transform(train['clean_comments'])

In [106]:
#Build TF-IDF features on train articles
# transform test articles into features
tv_test_features = tv_1_2.transform(test['clean_comments'])

In [107]:
from sklearn.svm import LinearSVC
scores = []
submission_svc = pd.DataFrame.from_dict({'id': test['id']})
submission_2_svc = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = train[class_name]
    classifier =LinearSVC(penalty='l2', C=1, random_state=42)

    cv_score = np.mean(cross_val_score(classifier, tv_train_features, train_target, cv=5, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(tv_train_features, train_target)
    submission_svc[class_name] = classifier.decision_function(tv_test_features)
    submission_2_svc[class_name] = classifier.predict(tv_test_features)

print('Total CV score is {}'.format(np.mean(scores)))

submission_svc.to_csv('submission_2_svc.csv', index=False)

CV score for class toxic is 0.9721529137848112
CV score for class severe_toxic is 0.9730849463128264
CV score for class obscene is 0.9831474737050507
CV score for class threat is 0.9850597517038615
CV score for class insult is 0.975242285292242
CV score for class identity_hate is 0.9642310019683895
Total CV score is 0.9754863954611969


# With Upsampling

In [109]:
train.head()

,index,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,clean_comments,Normalized_Comments
0,0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,explanation why the edits made under my userna...,explanation edits made username hardcore metal...
1,1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,daww he matches this background colour im seem...,daww matches background colour im seemingly st...
2,2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,hey man im really not trying to edit war its j...,hey man im really trying edit war guy constant...
3,3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,more cant make any real suggestions on improve...,cant make real suggestions improvement wondere...
4,4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,you sir are my hero any chance you remember wh...,sir hero chance remember page thats


In [111]:
# Let's create another column "bad_comment" denoting any type of toxicity in the comment

train["bad_comment"] = train.iloc[:,2:].sum(axis = 1)
train["bad_comment"][train["bad_comment"] != 0] = 1

In [112]:
train["bad_comment"].value_counts()

0    143346
1     16225
Name: bad_comment, dtype: int64

In [113]:
from sklearn.utils import resample

# Separate majority and minority classes
df_majority = train[train.bad_comment==0]
df_minority = train[train.bad_comment==1]
 
# Upsample minority class
df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=143346,    # to match majority class
                                 random_state=123) # reproducible results
 
# Combine majority class with upsampled minority class
df_upsampled = pd.concat([df_majority, df_minority_upsampled])
 
# Display new class counts
df_upsampled.bad_comment.value_counts()

1    143346
0    143346
Name: bad_comment, dtype: int64

In [115]:
#Build TF-IDF features on train articles
up_tv = TfidfVectorizer(use_idf=True, min_df=0.0, max_df=1.0)
up_tv_train_features = up_tv.fit_transform(df_upsampled["clean_comments"])

# transform test articles into features
up_tv_test_features = up_tv.transform(test["clean_comments"])

In [120]:
from sklearn.svm import LinearSVC
scores = []
submission_svc = pd.DataFrame.from_dict({'id': test['id']})
submission_2_svc = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = df_upsampled[class_name]
    classifier =LinearSVC(penalty='l2', C=1, random_state=42)

    cv_score = np.mean(cross_val_score(classifier, up_tv_train_features, train_target, cv=5, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(up_tv_train_features, train_target)
    submission_svc[class_name] = classifier.decision_function(up_tv_test_features)
    submission_2_svc[class_name] = classifier.predict(up_tv_test_features)

print('Total CV score is {}'.format(np.mean(scores)))

submission_svc.to_csv('submission_2_svc_up.csv', index=False)

CV score for class toxic is 0.9882514165815068
CV score for class severe_toxic is 0.9970261676752589
CV score for class obscene is 0.9976324589413077
CV score for class threat is 0.9999116387624365
CV score for class insult is 0.9939037743260434
CV score for class identity_hate is 0.999371619730338
Total CV score is 0.9960161793361486


In [125]:
scores = []
submission_lr = pd.DataFrame.from_dict({'id': test['id']})
submission_lr_2 = pd.DataFrame.from_dict({'id': test['id']})
for class_name in class_names:
    train_target = df_upsampled[class_name]
    classifier = LogisticRegression(C=0.1, solver='sag')

    cv_score = np.mean(cross_val_score(classifier, up_tv_train_features, train_target, cv=5, scoring='roc_auc'))
    scores.append(cv_score)
    print('CV score for class {} is {}'.format(class_name, cv_score))

    classifier.fit(up_tv_train_features, train_target)
    submission_lr[class_name] = classifier.predict_proba(up_tv_test_features)[:,1]
    submission_lr_2[class_name] = classifier.predict(up_tv_test_features)

print('Total CV score is {}'.format(np.mean(scores)))

submission.to_csv('submission_2_lr_up.csv', index=False)

CV score for class toxic is 0.9604113534502942
CV score for class severe_toxic is 0.9584155636068938
CV score for class obscene is 0.9653707950701547
CV score for class threat is 0.9846033090080096
CV score for class insult is 0.9421364662294103
CV score for class identity_hate is 0.9605032485736155
Total CV score is 0.961906789323063
